# 02 · Chunking, embedding and indexing

```
papers.jsonl ──► mixed chunking ──► chunks.jsonl
                      │
                      ▼
              BGE-large embeddings (GPU) ──► chunk_embeddings.npy
                      │
                      ▼
              Weaviate collection "ArxivChunk"
                • one vector config (self-provided vectors, HNSW, cosine)
                • BM25 index on title + text
                • filterable metadata (year, categories, date)
```

Each stage caches its output on disk, so later stages can be re-run independently.

In [1]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from common import *

papers = pd.read_json(PAPERS_PATH, lines=True, dtype={"arxiv_id": str, "submitted": str}).to_dict("records")
print(f"{len(papers):,} papers loaded from {PAPERS_PATH.name}")

122,346 papers loaded from papers.jsonl


## Mixed chunking
Sentence-based (variable-size) chunking with a fixed-size fallback and 10% overlap
(`CHUNK_WORDS = 120`, `OVERLAP_WORDS = 12`):

1. Abstracts of ≤ 120 words are kept as a single chunk.
2. Text is split into sentences.
3. Sentences longer than `CHUNK_WORDS − OVERLAP_WORDS` are cut into fixed-size windows.
4. Units are packed greedily into chunks of at most 120 words.
5. Each chunk starts with the last 12 words of the previous chunk.
6. A final chunk adding fewer than 20 new words is merged into the previous one.

```
 abstract:  [S1 ][S2      ][S3  ][S4 (very long sentence ...........)][S5]
 units:     [S1 ][S2      ][S3  ][S4a fixed][S4b fixed]               [S5]
 chunks:    [S1  S2  S3]
                    [ovl S4a]
                          [ovl S4b  S5]
```

Sentence boundaries keep chunks coherent, the fixed-size fallback bounds their length, and the overlap
keeps statements that span a boundary intact in at least one chunk.

In [2]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\[])")   # end punctuation followed by a capital/digit
MIN_TAIL_WORDS = 20

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]

def fixed_windows(words, size):
    """Fixed-size fallback: cut a word list into consecutive windows of `size` words."""
    return [words[i:i + size] for i in range(0, len(words), size)]

def mixed_chunk(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    if len(text.split()) <= size:                          # rule 1: short doc -> single chunk
        return [text]

    units = []                                             # rules 2 + 3
    for sent in split_sentences(text):
        words = sent.split()
        if len(words) > size - overlap:
            units += [" ".join(w) for w in fixed_windows(words, size - overlap)]
        else:
            units.append(sent)

    chunks, cur, new_words = [], [], 0                     # rules 4 + 5
    for unit in units:
        uw = unit.split()
        if new_words and len(cur) + len(uw) > size:
            chunks.append(cur)
            cur, new_words = cur[-overlap:], 0             # carry 10% overlap into the next chunk
        cur, new_words = cur + uw, new_words + len(uw)

    if chunks and new_words < MIN_TAIL_WORDS:              # rule 6: tiny-tail merge
        chunks[-1] = chunks[-1] + cur[len(cur) - new_words:]
    else:
        chunks.append(cur)
    return [" ".join(c) for c in chunks]

Example on a long abstract: the first 12 words of each chunk repeat the end of the previous chunk.

In [3]:
demo = max(papers[:2000], key=lambda p: len(p["abstract"].split()))
print(f"{demo['title']}\n({len(demo['abstract'].split())} words)\n")
for i, c in enumerate(mixed_chunk(demo["abstract"])):
    w = c.split()
    print(f"--- chunk {i}  ({len(w)} words)")
    print(f"  starts: ...{' '.join(w[:OVERLAP_WORDS])}")
    print(f"  ends  : {' '.join(w[-OVERLAP_WORDS:])}...")

Neural Fingerprints for Adversarial Attack Detection
(304 words)

--- chunk 0  (82 words)
  starts: ...Deep learning models for image classification have become standard tools in recent
  ends  : of two categories: (i) building robust classifiers (ii) directly detecting attacked images....
--- chunk 1  (100 words)
  starts: ...of two categories: (i) building robust classifiers (ii) directly detecting attacked images.
  ends  : propose to overcome this inherent limitation of any static defence with randomization....
--- chunk 2  (98 words)
  starts: ...propose to overcome this inherent limitation of any static defence with randomization.
  ends  : class they are considered a fingerprint and added to the detector bank....
--- chunk 3  (60 words)
  starts: ...class they are considered a fingerprint and added to the detector bank.
  ends  : model architectures, and show near-perfect detection with low rates of false detection....


### Full corpus
Each chunk carries its paper's metadata for filtering and citation. `chunk_id = "<arxiv_id>#<index>"`
is stable, so re-imports produce the same Weaviate UUIDs.

In [4]:
chunks = []
for p in tqdm(papers, desc="chunking"):
    pieces = mixed_chunk(p["abstract"])
    for i, piece in enumerate(pieces):
        chunks.append({
            "chunk_id": f"{p['arxiv_id']}#{i}",
            "arxiv_id": p["arxiv_id"],
            "title": p["title"],
            "text": piece,
            "chunk_index": i,
            "n_chunks": len(pieces),
            "authors": ", ".join(p["authors"][:10]) + (" et al." if len(p["authors"]) > 10 else ""),
            "categories": p["categories"],
            "primary_category": p["primary_category"],
            "published": p["submitted"],              # v1 submission month (YYYY-MM-01)
            "year": int(p["submitted"][:4]),
            "url": p["url"],
        })

with open(CHUNKS_PATH, "w") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

sizes = pd.Series([len(c["text"].split()) for c in chunks])
per_paper = Counter(c["n_chunks"] for c in chunks if c["chunk_index"] == 0)
print(f"{len(chunks):,} chunks from {len(papers):,} papers  ->  saved {CHUNKS_PATH.name}")
print("chunks per paper:", dict(sorted(per_paper.items())))
print("chunk size (words):", sizes.describe().round(1).to_dict())

chunking:   0%|          | 0/122346 [00:00<?, ?it/s]

257,025 chunks from 122,346 papers  ->  saved chunks.jsonl
chunks per paper: {1: 10498, 2: 89283, 3: 22302, 4: 261, 5: 1, 6: 1}
chunk size (words): {'count': 257025.0, 'mean': 92.5, 'std': 24.8, 'min': 14.0, '25%': 77.0, '50%': 101.0, '75%': 111.0, 'max': 139.0}


## Embeddings
- **Model:** `BAAI/bge-large-en-v1.5` (bi-encoder, 1024-d), fp16, batch size 256, GPU 0.
- **Input:** `title + "\n" + chunk_text`; the title gives later chunks of an abstract their context.
- **No instruction prefix on passages.** BGE's `QUERY_INSTRUCTION` is applied to queries only (notebook 03).
- **Normalised vectors**, so cosine similarity equals the dot product.
- Cached in `data/chunk_embeddings.npy`.

In [5]:
import torch
from sentence_transformers import SentenceTransformer

print("GPU:", torch.cuda.get_device_name(0))
embedder = SentenceTransformer(EMBED_MODEL, device="cuda").half()
embedder.max_seq_length = 256           # title + 120-word chunk fits comfortably

texts = [f"{c['title']}\n{c['text']}" for c in chunks]

if EMBEDDINGS_PATH.exists() and np.load(EMBEDDINGS_PATH, mmap_mode="r").shape[0] == len(texts):
    embeddings = np.load(EMBEDDINGS_PATH)
    print("loaded cached embeddings", embeddings.shape)
else:
    embeddings = embedder.encode(texts, batch_size=256, normalize_embeddings=True,
                                 show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    np.save(EMBEDDINGS_PATH, embeddings)
    print("saved", EMBEDDINGS_PATH.name, embeddings.shape,
          f"({EMBEDDINGS_PATH.stat().st_size / 1e9:.2f} GB)")

GPU: NVIDIA RTX 5000 Ada Generation


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

loaded cached embeddings (257025, 1024)


Sanity check: nearest chunks for a test query (brute force over a 20k sample).

In [6]:
q = embedder.encode(QUERY_INSTRUCTION + "parameter-efficient fine-tuning of large language models with low-rank adapters",
                    normalize_embeddings=True)
sims = embeddings[:20000] @ q
for i in np.argsort(-sims)[:3]:
    print(f"{sims[i]:.3f}  {chunks[i]['title']}")

0.861  Low-Rank Adaptation with Task-Relevant Feature Enhancement for Fine-tuning Language Models
0.856  Robust and Efficient Fine-tuning of LLMs with Bayesian Reparameterization of Low-Rank Adaptation
0.852  Initialization using Update Approximation is a Silver Bullet for Extremely Efficient Low-Rank Fine-Tuning


## Weaviate collection
Embedded Weaviate: the server runs as a child process of the Python client, with data persisted in
`weaviate_data/`.

| Property | Type | Used for |
|---|---|---|
| `title`, `text` | TEXT (word tokenisation) | BM25 keyword search (title boosted ×2 at query time) |
| `authors` | TEXT | display and keyword search |
| `categories` | TEXT[] (field tokenisation) | category filter |
| `primary_category`, `arxiv_id`, `chunk_id` | TEXT (field) | exact-match filters, grouping |
| `published` | DATE | date-range filter |
| `year`, `chunk_index`, `n_chunks` | INT | filters, bookkeeping |
| `url` | TEXT (not indexed) | citation link |

Vectors are self-provided (`Configure.Vectors.self_provided()`): a single HNSW index with cosine
distance and no vectorizer module, so the semantic side of hybrid search uses only the bi-encoder.

In [7]:
import weaviate.classes as wvc
from weaviate.classes.config import Configure, DataType, Property, Tokenization, VectorDistances
from weaviate.util import generate_uuid5

client = get_weaviate_client()
print("Weaviate ready:", client.is_ready(), "| version", client.get_meta()["version"])

if client.collections.exists(COLLECTION):
    client.collections.delete(COLLECTION)          # rebuild from scratch (idempotent notebook)

text_prop = lambda name, **kw: Property(name=name, data_type=DataType.TEXT, **kw)
field_prop = lambda name: Property(name=name, data_type=DataType.TEXT, tokenization=Tokenization.FIELD,
                                   index_searchable=False)

collection = client.collections.create(
    name=COLLECTION,
    vector_config=Configure.Vectors.self_provided(
        vector_index_config=Configure.VectorIndex.hnsw(
            distance_metric=VectorDistances.COSINE,
            ef_construction=128,          # build-time quality/speed trade-off
            max_connections=32,           # graph degree; higher = better recall, more RAM
        )
    ),
    inverted_index_config=Configure.inverted_index(bm25_k1=1.2, bm25_b=0.75),
    properties=[
        text_prop("title", tokenization=Tokenization.WORD),
        text_prop("text", tokenization=Tokenization.WORD),
        text_prop("authors", tokenization=Tokenization.WORD),
        Property(name="categories", data_type=DataType.TEXT_ARRAY, tokenization=Tokenization.FIELD,
                 index_searchable=False),
        field_prop("primary_category"),
        field_prop("arxiv_id"),
        field_prop("chunk_id"),
        Property(name="published", data_type=DataType.DATE),
        Property(name="year", data_type=DataType.INT),
        Property(name="chunk_index", data_type=DataType.INT),
        Property(name="n_chunks", data_type=DataType.INT),
        Property(name="url", data_type=DataType.TEXT, index_searchable=False, index_filterable=False),
    ],
)
print("created collection", COLLECTION)

Weaviate ready: True | version 1.40.2
created collection ArxivChunk


## Batch import
Objects and vectors are sent over gRPC with `batch.fixed_size` (1,000 per batch, 4 concurrent requests);
`generate_uuid5(chunk_id)` makes re-imports idempotent. Takes ~2 min for 257k objects.

The previous cell recreates the collection, so an interrupted import leaves a partial index: re-run
from the previous cell. The check below should report `257025` objects.

In [8]:
with collection.batch.fixed_size(batch_size=1000, concurrent_requests=4) as batch:
    for c, vec in tqdm(zip(chunks, embeddings), total=len(chunks), desc="importing"):
        batch.add_object(
            properties={**c, "published": c["published"] + "T00:00:00Z"},
            vector=vec.tolist(),
            uuid=generate_uuid5(c["chunk_id"]),
        )

failed = collection.batch.failed_objects
print(f"failed objects: {len(failed)}")
if failed:
    print(failed[0].message)
print("objects in collection:", collection.aggregate.over_all(total_count=True).total_count)

importing:   0%|          | 0/257025 [00:00<?, ?it/s]

failed objects: 0
objects in collection: 257025


## Search modes
Keyword, vector and hybrid search for the same query. `alpha` weights hybrid search between BM25
(`0`) and vector search (`1`).

In [9]:
from weaviate.classes.query import Filter, HybridFusion, MetadataQuery

query = "LoRA fine-tuning"
qvec = embedder.encode(QUERY_INSTRUCTION + query, normalize_embeddings=True).tolist()

for name, alpha in [("BM25 only", 0.0), ("vector only", 1.0), ("hybrid", 0.6)]:
    res = collection.query.hybrid(query=query, vector=qvec, alpha=alpha, limit=3,
                                  query_properties=["title^2", "text"],
                                  fusion_type=HybridFusion.RELATIVE_SCORE,
                                  filters=Filter.by_property("categories").contains_any(["cs.CL"]),
                                  return_metadata=MetadataQuery(score=True))
    print(f"\n[{name}]")
    for o in res.objects:
        print(f"  {o.metadata.score:.3f}  {o.properties['title'][:90]}")


[BM25 only]
  1.000  DR-LoRA: Dynamic Rank LoRA for Fine-Tuning Mixture-of-Experts Models
  0.939  Parameter-Efficient Fine-Tuning for Medical Text Summarization: A Comparative Study of Lor
  0.935  DR-LoRA: Dynamic Rank LoRA for Fine-Tuning Mixture-of-Experts Models

[vector only]
  1.000  PLoP: Precise LoRA Placement for Efficient Finetuning of Large Models
  0.829  Dynamic Adaptation of LoRA Fine-Tuning for Efficient and Task-Specific Optimization of Lar
  0.785  SMoA: Spectrum Modulation Adapter for Parameter-Efficient Fine-Tuning

[hybrid]
  0.659  LowRA: Accurate and Efficient LoRA Fine-Tuning of LLMs under 2 Bits
  0.638  SMoA: Spectrum Modulation Adapter for Parameter-Efficient Fine-Tuning
  0.600  PLoP: Precise LoRA Placement for Efficient Finetuning of Large Models


In [10]:
client.close()      # stops the embedded server; data stays on disk for step 3
print("done; next: 03_rag_pipeline.ipynb")

done; next: 03_rag_pipeline.ipynb
